# 01 · Why Pydantic? `BaseModel` Basics

**Module 15 – Pydantic: Data Validation & Settings** · Estimated time: 45 min

> **Pydantic** is the most widely used data-validation library in Python (FastAPI, LangChain, the OpenAI/Anthropic SDKs, Django Ninja, SQLModel and thousands of projects use it). Its core is written in **Rust** (`pydantic-core`), so it's very fast. This module teaches **Pydantic v2** from scratch.
>
> **Prerequisites:** type hints (Module 05 · 06), classes & dataclasses (Module 06), exceptions (Module 07 · 01), JSON (Module 11 · 03).

### 🎯 What you will learn
- The problem Pydantic solves: **untrusted, messy data** at your program's boundaries
- Defining models with type hints; creating instances; **validation errors**
- Type **coercion** ("lax mode"): `"42"` → `42`
- Required vs optional fields, defaults (mutable defaults are safe!)
- The core methods: `model_validate`, `model_validate_json`, `model_dump`, `model_dump_json`, `model_copy`, `model_fields`
- Pydantic vs dataclasses vs `TypedDict`

In [1]:
import pydantic
print("Pydantic", pydantic.VERSION)

Pydantic 2.14.1


## 1. The problem

Data coming from **outside** your code (HTTP requests, JSON files, environment variables, CSV, LLM outputs, user forms) can be missing fields, have wrong types, or contain garbage. Checking it by hand is long and error-prone:

In [2]:
def create_user_manual(data: dict) -> dict:
    if "name" not in data or not isinstance(data["name"], str) or not data["name"].strip():
        raise ValueError("name is required")
    try:
        age = int(data.get("age"))
    except (TypeError, ValueError):
        raise ValueError("age must be an integer") from None
    if not 0 <= age <= 150:
        raise ValueError("age must be between 0 and 150")
    tags = data.get("tags", [])
    if not isinstance(tags, list) or not all(isinstance(t, str) for t in tags):
        raise ValueError("tags must be a list of strings")
    return {"name": data["name"].strip(), "age": age, "tags": tags}

print(create_user_manual({"name": " Asha ", "age": "31"}))

{'name': 'Asha', 'age': 31, 'tags': []}


With Pydantic you **declare** the shape with type hints, and validation, conversion, error messages, serialisation and JSON Schema come for free.

## 2. Your first model

In [3]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str
    age: int | None = None             # optional, default None
    tags: list[str] = []               # ✅ safe: Pydantic copies mutable defaults per instance

user = User(id=1, name="Asha", email="asha@example.com", age="31")   # note: age given as a STRING
print(user)
print(user.age, type(user.age))        # coerced to int
print(repr(user))

id=1 name='Asha' email='asha@example.com' age=31 tags=[]
31 <class 'int'>
User(id=1, name='Asha', email='asha@example.com', age=31, tags=[])


## 3. Validation errors

In [4]:
from pydantic import ValidationError

try:
    User(id="abc", name=None, email="x@y.z", tags=["ok", 42])
except ValidationError as e:
    print(e.error_count(), "errors")
    print(e)

3 errors
3 validation errors for User
id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='abc', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_parsing
name
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
tags.1
  Input should be a valid string [type=string_type, input_value=42, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type


Each error tells you **where** (`loc`), **what** (`type`, `msg`) and **the bad input**. As structured data:

In [5]:
try:
    User(id="abc", email="x@y.z")
except ValidationError as e:
    for err in e.errors():
        print(err["loc"], err["type"], "→", err["msg"], "| input:", err["input"])

('id',) int_parsing → Input should be a valid integer, unable to parse string as an integer | input: abc
('name',) missing → Field required | input: {'id': 'abc', 'email': 'x@y.z'}


Pydantic validates **all** fields and reports **every** problem at once, which is far friendlier than failing on the first.

## 4. Coercion: lax (default) vs strict

By default Pydantic converts sensible inputs: `"42"` → `42`, `"true"`/`1` → `True`, `"2026-10-20"` → `date`, `4.0` → `4`. It refuses lossy or nonsensical ones:

In [6]:
from datetime import date

class Event(BaseModel):
    count: int
    active: bool
    day: date
    price: float

print(Event(count="42", active="yes", day="2026-10-20", price="9.99"))
for bad in [{"count": "4.5"}, {"count": 4.5}, {"active": "maybe"}, {"day": "20/10/2026"}]:
    try:
        Event(**{"count": 1, "active": True, "day": "2026-10-20", "price": 1.0, **bad})
    except ValidationError as e:
        print(f"{bad} → {e.errors()[0]['msg']}")

count=42 active=True day=datetime.date(2026, 10, 20) price=9.99
{'count': '4.5'} → Input should be a valid integer, unable to parse string as an integer
{'count': 4.5} → Input should be a valid integer, got a number with a fractional part
{'active': 'maybe'} → Input should be a valid boolean, unable to interpret input
{'day': '20/10/2026'} → Input should be a valid date or datetime, invalid character in year


Strict mode (no coercion at all) is covered in lesson 06.

## 5. Required vs optional fields

| Declaration | Required? | Can be `None`? |
|---|---|---|
| `name: str` | ✅ | ❌ |
| `name: str = "x"` | ❌ (default `"x"`) | ❌ |
| `name: str \| None` | ✅ (must be given, may be `None`) | ✅ |
| `name: str \| None = None` | ❌ | ✅ |

In [7]:
class Profile(BaseModel):
    username: str
    nickname: str | None              # required, but may be None!
    bio: str = ""
    website: str | None = None

try:
    Profile(username="asha")
except ValidationError as e:
    print(e.errors()[0]["loc"], e.errors()[0]["msg"])
print(Profile(username="asha", nickname=None))

('nickname',) Field required
username='asha' nickname=None bio='' website=None


## 6. The core API

| Method | Purpose |
|---|---|
| `Model(**data)` | create & validate from keyword args |
| `Model.model_validate(obj)` | validate a dict (or object with `from_attributes`) |
| `Model.model_validate_json(text)` | parse **and** validate JSON in one fast step |
| `m.model_dump()` | → `dict` (Python types) |
| `m.model_dump(mode="json")` | → JSON-compatible `dict` (dates → strings...) |
| `m.model_dump_json()` | → JSON string |
| `m.model_copy(update={...})` | copy with changes (`deep=True` for deep copy) |
| `Model.model_fields` | field definitions |
| `Model.model_json_schema()` | JSON Schema (lesson 08) |
| `Model.model_construct(**data)` | create **without** validation (trusted data only) |

In [8]:
raw_json = '{"id": 7, "name": "Ravi", "email": "ravi@example.com", "tags": ["admin"]}'
u = User.model_validate_json(raw_json)
print(u)
print(User.model_validate({"id": 8, "name": "Meena", "email": "m@example.com"}))

print(u.model_dump())
print(u.model_dump_json())
print(u.model_dump(include={"id", "name"}), u.model_dump(exclude={"tags"}))

u2 = u.model_copy(update={"name": "Ravi Kumar"})
print(u2.name, "| original unchanged:", u.name)

print({name: (f.annotation, f.is_required()) for name, f in User.model_fields.items()})

id=7 name='Ravi' email='ravi@example.com' age=None tags=['admin']
id=8 name='Meena' email='m@example.com' age=None tags=[]
{'id': 7, 'name': 'Ravi', 'email': 'ravi@example.com', 'age': None, 'tags': ['admin']}
{"id":7,"name":"Ravi","email":"ravi@example.com","age":null,"tags":["admin"]}
{'id': 7, 'name': 'Ravi'} {'id': 7, 'name': 'Ravi', 'email': 'ravi@example.com', 'age': None}
Ravi Kumar | original unchanged: Ravi
{'id': (<class 'int'>, True), 'name': (<class 'str'>, True), 'email': (<class 'str'>, True), 'age': (int | None, False), 'tags': (list[str], False)}


> ⚠️ `model_copy(update=...)` does **not** validate the update. Use `Model.model_validate({**m.model_dump(), **changes})` if the changes come from outside.

Models support `==` (compares fields), iteration over `(name, value)` pairs and attribute assignment (without validation unless configured, lesson 06):

In [9]:
print(u == User.model_validate_json(raw_json))
print(dict(u))
u.age = 40
print(u.age)

True
{'id': 7, 'name': 'Ravi', 'email': 'ravi@example.com', 'age': None, 'tags': ['admin']}
40


## 7. Pydantic vs dataclasses vs TypedDict

| | `@dataclass` | `TypedDict` | Pydantic `BaseModel` |
|---|---|---|---|
| Runtime validation | ❌ | ❌ | ✅ |
| Type coercion | ❌ | ❌ | ✅ (configurable) |
| JSON parse / dump | manual | manual | ✅ built in |
| JSON Schema | ❌ | ❌ | ✅ |
| Speed | fastest to create | plain dict | fast (Rust core), validation costs a bit |
| Best for | internal data structures | typing dicts | **data at boundaries**: APIs, configs, files, LLM output |

✅ Rule of thumb: validate data **once, at the edges** of your system with Pydantic; pass validated objects (or dataclasses) internally.

## 📝 Summary
- Declare models with type hints; Pydantic validates, coerces and reports all errors in a `ValidationError`.
- Required vs optional is about **defaults**; `X | None` only allows `None`.
- Core API: `model_validate(_json)`, `model_dump(_json)`, `model_copy`, `model_fields`.

## 🏋️ Practice
1. Create a `Product` model with `sku: str`, `name: str`, `price: float`, `stock: int = 0`, `tags: list[str] = []`. Validate `{"sku": "A1", "name": "Pen", "price": "12.5"}`.
2. Feed it invalid data with three problems and print each error's `loc` and `msg`.
3. Parse a JSON array of products using a loop over `json.loads` + `model_validate`, then compare with lesson 07's `TypeAdapter(list[Product])`.

<details><summary>Show solutions</summary>

```python
from pydantic import BaseModel, ValidationError

class Product(BaseModel):
    sku: str
    name: str
    price: float
    stock: int = 0
    tags: list[str] = []

print(Product.model_validate({"sku": "A1", "name": "Pen", "price": "12.5"}))
try:
    Product(sku=1, price="free", stock="many")
except ValidationError as e:
    for err in e.errors():
        print(err["loc"], err["msg"])
```
</details>